# CLAW summary tables and control ensemble changes

This notebook writes the tables used by the following figures:

1. State-level CLAW summaries: left-choice probability, decision time, and terminal probability.
2. Outgoing transition probabilities, including END transitions.
3. Changes in CBGT control ensemble and DDM parameters along observed state transitions.

Run Notebooks 03 and 05 to save the clustering labels, followed by Notebooks 07 and 08 to audit those assigned units. This notebook then joins each audit to the saved labels, removes units marked strongly FSI-like, and keeps every other unit with its original label. It rebuilds the empirical population-activity tables and calculates the CLAWs for every session. It does not rerun clustering or change the label of any retained unit.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from one.alf.exceptions import ALFWarning
from one.api import ONE

from spn_figures.config import DERIVED, SOURCE
from spn_figures.io import read_csv, write_csv
from spn_figures.datasets import load_steinmetz_session_exact
from spn_figures.clustering import (
    apply_fsi_audit_to_fixed_labels,
    ibl_population_activity_table_exact,
    steinmetz_population_activity_table_exact,
)
from spn_figures.claw import (
    binarize_activity,
    build_cbgt_state_activity,
    control_transition_scores,
    fit_control_ensembles,
    summarize_claw,
    transition_table_with_end,
    trial_sequences,
)

warnings.filterwarnings("ignore", category=ALFWarning, message="Multiple revisions.*")

## 1. Build trial-level state walks

CBGT state bins are already binary. For each empirical dataset, the waveform audit is joined to the original fixed clustering labels. Only strongly FSI-like units are removed; all remaining units keep their original subtype labels. The resulting population activity is then binarized session by session using the original CLAW logic.

In [3]:
population_order = [
    "dSPN_left",
    "dSPN_right",
    "iSPN_left",
    "iSPN_right",
]

stein_labels = read_csv(DERIVED / "steinmetz" / "unit_labels.csv")
stein_audit = read_csv(
    DERIVED / "steinmetz" / "postclustering_waveform_audit.csv"
)
stein_labels["unit_id"] = stein_labels["unit_id"].astype(int)
stein_audit["unit_id"] = stein_audit["unit_id"].astype(int)
stein_fixed_labels, stein_summary = apply_fsi_audit_to_fixed_labels(
    stein_labels,
    stein_audit,
    match_columns=["session", "unit_id"],
    minimum_units_per_population=1,
)

ibl_labels = read_csv(DERIVED / "ibl" / "unit_labels.csv")
ibl_audit = read_csv(
    DERIVED / "ibl" / "postclustering_spn_fsi_waveform_audit.csv"
)
for frame in [ibl_labels, ibl_audit]:
    frame["pid"] = frame["pid"].astype(str)
    frame["unit_id"] = frame["unit_id"].astype(int)
ibl_fixed_labels, ibl_summary = apply_fsi_audit_to_fixed_labels(
    ibl_labels,
    ibl_audit,
    match_columns=["session", "pid", "unit_id"],
    minimum_units_per_population=1,
)

# Save an explicit unit-level record of what is retained and used.
write_csv(
    stein_fixed_labels,
    DERIVED / "steinmetz" / "post_audit_fixed_unit_labels.csv",
)
write_csv(
    ibl_fixed_labels,
    DERIVED / "ibl" / "post_audit_fixed_unit_labels.csv",
)
write_csv(
    stein_summary,
    DERIVED / "steinmetz" / "claw_session_summary.csv",
)
write_csv(
    ibl_summary,
    DERIVED / "ibl" / "claw_session_summary.csv",
)

stein_summary_for_display = stein_summary.copy()
stein_summary_for_display.insert(0, "dataset", "Steinmetz")
ibl_summary_for_display = ibl_summary.copy()
ibl_summary_for_display.insert(0, "dataset", "IBL")
fsi_filter_summary = pd.concat(
    [ibl_summary_for_display, stein_summary_for_display],
    ignore_index=True,
)
provenance_columns = [
    "dataset",
    "session",
    "fixed_labels_preserved",
    "n_units_excluded_as_fsi",
    "n_units_for_claw",
    "n_dSPN_left_for_claw",
    "n_dSPN_right_for_claw",
    "n_iSPN_left_for_claw",
    "n_iSPN_right_for_claw",
]
provenance_path = DERIVED / "combined" / "fsi_exclusion_summary.csv"
write_csv(fsi_filter_summary[provenance_columns], provenance_path)
print("Fixed-label post-audit unit counts for the empirical CLAWs:")
display(fsi_filter_summary[provenance_columns])

# Rebuild Steinmetz population activity from the saved labels.
stein_manifest = read_csv(SOURCE / "steinmetz" / "session_manifest.csv")
manifest_sessions = set(stein_manifest["session"].astype(str))
label_sessions = set(stein_fixed_labels["session"].astype(str))
if manifest_sessions != label_sessions:
    raise ValueError(
        "Steinmetz manifest sessions do not match the fixed-label table."
    )

steinmetz_activities = []
for row in stein_manifest.itertuples(index=False):
    session_labels = stein_fixed_labels.loc[
        stein_fixed_labels["session"].astype(str).eq(str(row.session))
    ].copy()
    payload = load_steinmetz_session_exact(
        SOURCE / "steinmetz" / row.folder,
        session_name=row.session,
        matching_seed=int(row.matching_seed),
        premove_gap_s=float(row.premove_gap_s),
    )
    fixed_result = {
        "session": str(row.session),
        "source_payload": payload,
        "unit_ids": session_labels["unit_id"].to_numpy(int),
        "final_labels": session_labels["final_label"].to_numpy(int),
        "final_names": session_labels["final_name"].to_numpy(object),
    }
    excluded_unit_ids = session_labels.loc[
        session_labels["strongly_fsi_like"], "unit_id"
    ].to_numpy(int)
    steinmetz_activities.append(
        steinmetz_population_activity_table_exact(
            fixed_result,
            minimum_units_per_population=1,
            excluded_unit_ids=excluded_unit_ids,
        )
    )

if not steinmetz_activities:
    raise RuntimeError("No Steinmetz population activity was generated.")
steinmetz_activity = pd.concat(steinmetz_activities, ignore_index=True)
steinmetz_activity.to_csv(
    DERIVED / "steinmetz" / "population_activity_bins.csv.gz",
    index=False,
)

# Rebuild IBL population activity from the saved labels.
CACHE_DIR = Path.home() / "ibl_bwm_cache"
ONE.setup(
    base_url="https://openalyx.internationalbrainlab.org",
    cache_dir=CACHE_DIR,
    silent=True,
)
one = ONE(password="international")
try:
    one.load_cache(tag="Brainwidemap")
except Exception:
    pass

ibl_activities = []
for (session, pid), session_labels in ibl_fixed_labels.groupby(
    ["session", "pid"], sort=False
):
    metadata_columns = ["eid", "probe"]
    if any(session_labels[column].nunique() != 1 for column in metadata_columns):
        raise ValueError(f"{session}: inconsistent saved IBL insertion metadata.")
    fixed_result = {
        "session": str(session),
        "eid": str(session_labels["eid"].iloc[0]),
        "pid": str(pid),
        "probe": str(session_labels["probe"].iloc[0]),
        "unit_ids": session_labels["unit_id"].to_numpy(int),
        "final_labels": session_labels["final_label"].to_numpy(int),
        "final_names": session_labels["final_name"].to_numpy(object),
    }
    excluded_unit_ids = session_labels.loc[
        session_labels["strongly_fsi_like"], "unit_id"
    ].to_numpy(int)
    ibl_activities.append(
        ibl_population_activity_table_exact(
            fixed_result,
            one=one,
            minimum_units_per_population=1,
            excluded_unit_ids=excluded_unit_ids,
        )
    )

if not ibl_activities:
    raise RuntimeError("No IBL population activity was generated.")
ibl_activity = pd.concat(ibl_activities, ignore_index=True)
ibl_activity.to_csv(
    DERIVED / "ibl" / "population_activity_bins.csv.gz",
    index=False,
)

# CBGT: states are already binary in the data generated by Notebook 01.
cbgt_bins = read_csv(DERIVED / "cbgt" / "trial_state_bins.csv.gz")
cbgt_bins["dataset"] = "CBGT"
if "state" not in cbgt_bins.columns:
    cbgt_bins["state"] = sum(
        cbgt_bins[name].astype(int) * (2 ** (3 - index))
        for index, name in enumerate(["dSPN_left", "dSPN_right", "iSPN_left", "iSPN_right"])
    ).astype(int)

# Empirical datasets: binarize the fixed-label, FSI-filtered activity above.
ibl_activity["dataset"] = "IBL"
ibl_bins, ibl_thresholds = binarize_activity(ibl_activity)

steinmetz_activity["dataset"] = "Steinmetz"
steinmetz_bins, steinmetz_thresholds = binarize_activity(steinmetz_activity)

# Save the empirical binary time-bin tables and thresholds.
ibl_bins.to_csv(DERIVED / "ibl" / "state_bins.csv.gz", index=False)
steinmetz_bins.to_csv(DERIVED / "steinmetz" / "state_bins.csv.gz", index=False)
write_csv(ibl_thresholds, DERIVED / "ibl" / "binarization_thresholds.csv")
write_csv(steinmetz_thresholds, DERIVED / "steinmetz" / "binarization_thresholds.csv")

# Convert binned state tables to compressed trial walks.
cbgt_trials = trial_sequences(cbgt_bins)
ibl_trials = trial_sequences(ibl_bins)
steinmetz_trials = trial_sequences(steinmetz_bins)

cbgt_trials.to_csv(DERIVED / "cbgt" / "trial_sequences.csv.gz", index=False)
ibl_trials.to_csv(DERIVED / "ibl" / "trial_sequences.csv.gz", index=False)
steinmetz_trials.to_csv(DERIVED / "steinmetz" / "trial_sequences.csv.gz", index=False)

print("Trial sequences:")
print("CBGT:", len(cbgt_trials))
print("IBL:", len(ibl_trials))
print("Steinmetz:", len(steinmetz_trials))

Fixed-label post-audit unit counts for the empirical CLAWs:


,dataset,session,fixed_labels_preserved,n_units_excluded_as_fsi,n_units_for_claw,n_dSPN_left_for_claw,n_dSPN_right_for_claw,n_iSPN_left_for_claw,n_iSPN_right_for_claw
0,IBL,churchlandlab | CSHL049 | 2020-01-08,True,12,12,3,3,5,1
1,IBL,cortexlab | KS086 | 2022-03-16,True,4,19,5,2,4,8
2,IBL,danlab | DY_014 | 2020-07-18,True,2,27,4,6,7,10
3,IBL,hoferlab | SWC_061 | 2020-11-23,True,0,24,9,3,8,4
4,IBL,mrsicflogellab | SWC_038 | 2020-07-31,True,0,16,3,3,5,5
5,IBL,mrsicflogellab | SWC_038 | 2020-08-01,True,0,35,3,26,3,3
6,IBL,zadorlab | CSH_ZAD_026 | 2020-08-17,True,3,26,13,2,3,8
7,Steinmetz,Hench_2017-06-18,True,0,50,3,24,19,4
8,Steinmetz,Lederberg_2017-12-11,True,1,28,3,9,12,4
9,Steinmetz,Radnitz_2017-01-12,True,5,45,6,23,13,3


Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"
Downloading: /Users/zhuojunyu/ibl_bwm_cache/Brainwidemap/tmpdeqg5oif/cache.zip Bytes: 4353173


100%|██████████| 4.1515092849731445/4.1515092849731445 [00:00<00:00, 21.80it/s]


Trial sequences:
CBGT: 15000
IBL: 752
Steinmetz: 386


## 2. Output state metrics and transition probabilities

`claw_nodes.csv` contains one row per state. `claw_transitions_with_end.csv` contains all outgoing transition probabilities, including transitions to `END`.

In [4]:
cbgt_nodes, cbgt_edges = summarize_claw(cbgt_trials, balance_choice=False)
ibl_nodes, ibl_edges = summarize_claw(ibl_trials, balance_choice=True)
steinmetz_nodes, steinmetz_edges = summarize_claw(steinmetz_trials, balance_choice=True)

summary_tables = {
    "cbgt": (cbgt_nodes, cbgt_edges, DERIVED / "cbgt"),
    "ibl": (ibl_nodes, ibl_edges, DERIVED / "ibl"),
    "steinmetz": (steinmetz_nodes, steinmetz_edges, DERIVED / "steinmetz"),
}

for name, (nodes, edges, folder) in summary_tables.items():
    transitions = transition_table_with_end(nodes, edges)
    write_csv(nodes, folder / "claw_nodes.csv")
    write_csv(edges, folder / "claw_edges.csv")
    write_csv(transitions, folder / "claw_transitions_with_end.csv")

    print(f"\n{name.upper()} state metrics")
    display(
        nodes[
            [
                "state",
                "bits",
                "n_trials",
                "left_choice_probability",
                "mean_decision_time_ms",
                "median_decision_time_ms",
                "terminal_probability",
                "end_count",
                "outgoing_total",
            ]
        ].sort_values("state")
    )

    print(f"{name.upper()} outgoing transitions, including END")
    display(transitions)


CBGT state metrics


,state,bits,n_trials,left_choice_probability,mean_decision_time_ms,median_decision_time_ms,terminal_probability,end_count,outgoing_total
0,0,"[0,0,0,0]",14999,0.504700,115.141009,100.0,0.076983,1191,15471
1,1,"[0,0,0,1]",1142,0.407180,170.551664,150.0,0.196594,254,1292
2,2,"[0,0,1,0]",1184,0.559966,169.290541,150.0,0.161265,209,1296
3,3,"[0,0,1,1]",1289,0.476338,181.737781,160.0,0.283007,448,1583
4,4,"[0,1,0,0]",4281,0.129175,110.665732,100.0,0.546377,2368,4334
5,5,"[0,1,0,1]",1464,0.119536,134.521858,110.0,0.682243,1022,1498
6,6,"[0,1,1,0]",234,0.243590,148.675214,130.0,0.412500,99,240
7,7,"[0,1,1,1]",1078,0.218924,175.519481,150.0,0.588380,719,1222
8,8,"[1,0,0,0]",4269,0.880300,111.091591,100.0,0.526085,2279,4332
9,9,"[1,0,0,1]",260,0.715385,146.692308,120.0,0.397727,105,264


CBGT outgoing transitions, including END


,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,1191,0.076983
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,1061,0.068580
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,1105,0.071424
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,367,0.023722
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,4231,0.273479
...,...,...,...,...,...,...,...
202,15,"[1,1,1,1]",10,"[1,0,1,0]",state_to_state,5,0.003228
203,15,"[1,1,1,1]",11,"[1,0,1,1]",state_to_state,120,0.077469
204,15,"[1,1,1,1]",12,"[1,1,0,0]",state_to_state,2,0.001291
205,15,"[1,1,1,1]",13,"[1,1,0,1]",state_to_state,16,0.010329



IBL state metrics


...

,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,411,0.296110
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,197,0.141931
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,246,0.177233
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,7,0.005043
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,190,0.136888
...,...,...,...,...,...,...,...
99,12,"[1,1,0,0]",14,"[1,1,1,0]",state_to_state,1,0.040000
100,13,"[1,1,0,1]",1,"[0,0,0,1]",state_to_state,1,0.166667
101,13,"[1,1,0,1]",9,"[1,0,0,1]",state_to_state,2,0.333333
102,13,"[1,1,0,1]",12,"[1,1,0,0]",state_to_state,3,0.500000



STEINMETZ state metrics


,state,bits,n_trials,left_choice_probability,mean_decision_time_ms,median_decision_time_ms,terminal_probability,end_count,outgoing_total
0,0,"[0,0,0,0]",386,0.500000,207.272451,189.452258,0.215278,155,720
1,1,"[0,0,0,1]",68,0.352941,225.835292,207.774918,0.239130,22,92
2,2,"[0,0,1,0]",159,0.603774,220.481080,205.727750,0.347222,75,216
3,3,"[0,0,1,1]",12,0.416667,242.658731,213.732587,0.153846,2,13
4,4,"[0,1,0,0]",126,0.253968,222.270102,207.261933,0.268293,44,164
5,5,"[0,1,0,1]",27,0.148148,238.116553,216.789043,0.366667,11,30
6,6,"[0,1,1,0]",57,0.263158,223.365908,213.787505,0.454545,30,66
7,7,"[0,1,1,1]",4,0.000000,273.512046,234.664544,0.250000,1,4
8,8,"[1,0,0,0]",121,0.512397,220.351456,212.478034,0.102273,18,176
9,9,"[1,0,0,1]",8,0.750000,208.761301,193.610690,0.250000,2,8


STEINMETZ outgoing transitions, including END


,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,155,0.215278
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,78,0.108333
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,165,0.229167
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,4,0.005556
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,131,0.181944
...,...,...,...,...,...,...,...
90,13,"[1,1,0,1]",5,"[0,1,0,1]",state_to_state,1,0.200000
91,13,"[1,1,0,1]",12,"[1,1,0,0]",state_to_state,1,0.200000
92,14,"[1,1,1,0]",END,END,state_to_END,3,0.750000
93,14,"[1,1,1,0]",6,"[0,1,1,0]",state_to_state,1,0.250000


### Pooled IBL + Steinmetz CLAW statistics

All eligible trial sequences from every retained IBL and Steinmetz session are combined here. Session names are prefixed by dataset to preserve unique session identities. The pooled CLAW uses the same within-session choice balancing as the separate empirical CLAWs; decision times and transition/terminal probabilities use all pooled trials.

In [5]:
ibl_trials_pooled = ibl_trials.copy()
ibl_trials_pooled["session"] = "IBL::" + ibl_trials_pooled["session"].astype(str)

steinmetz_trials_pooled = steinmetz_trials.copy()
steinmetz_trials_pooled["session"] = (
    "Steinmetz::" + steinmetz_trials_pooled["session"].astype(str)
)

empirical_trials = pd.concat(
    [ibl_trials_pooled, steinmetz_trials_pooled],
    ignore_index=True,
)
empirical_trials["dataset"] = "IBL+Steinmetz"

if len(empirical_trials) != len(ibl_trials) + len(steinmetz_trials):
    raise RuntimeError("The pooled empirical trial table is incomplete.")

empirical_nodes, empirical_edges = summarize_claw(
    empirical_trials,
    balance_choice=True,
)
empirical_transitions = transition_table_with_end(
    empirical_nodes,
    empirical_edges,
)

combined_dir = DERIVED / "combined"
write_csv(empirical_nodes, combined_dir / "empirical_claw_nodes.csv")
write_csv(empirical_edges, combined_dir / "empirical_claw_edges.csv")
write_csv(
    empirical_transitions,
    combined_dir / "empirical_claw_transitions_with_end.csv",
)

print(
    f"Pooled empirical CLAW: {len(empirical_trials)} trials from "
    f"{empirical_trials['session'].nunique()} sessions"
)
display(
    empirical_nodes[
        [
            "state",
            "bits",
            "n_trials",
            "left_choice_probability",
            "mean_decision_time_ms",
            "median_decision_time_ms",
            "terminal_probability",
            "end_count",
            "outgoing_total",
        ]
    ].sort_values("state")
)

print("IBL+STEINMETZ outgoing transitions, including END")
display(empirical_transitions)

Pooled empirical CLAW: 1138 trials from 11 sessions


,state,bits,n_trials,left_choice_probability,mean_decision_time_ms,median_decision_time_ms,terminal_probability,end_count,outgoing_total
0,0,"[0,0,0,0]",1135,0.499559,215.447450,203.009272,0.268501,566,2108
1,1,"[0,0,0,1]",212,0.457547,239.252892,225.189918,0.239645,81,338
2,2,"[0,0,1,0]",363,0.556474,231.736089,216.453077,0.262548,136,518
3,3,"[0,0,1,1]",40,0.375000,252.914369,221.369630,0.267857,15,56
4,4,"[0,1,0,0]",280,0.382143,239.108460,232.267147,0.201018,79,393
5,5,"[0,1,0,1]",46,0.282609,249.568208,235.175557,0.320755,17,53
6,6,"[0,1,1,0]",88,0.272727,233.539332,219.595280,0.390909,43,110
7,7,"[0,1,1,1]",13,0.230769,259.410503,230.244119,0.285714,4,14
8,8,"[1,0,0,0]",351,0.586895,231.883533,220.812778,0.197740,105,531
9,9,"[1,0,0,1]",45,0.622222,224.585995,223.114042,0.314815,17,54


IBL+STEINMETZ outgoing transitions, including END


,source,source_bits,target,target_bits,transition_type,count,probability
0,0,"[0,0,0,0]",END,END,state_to_END,566,0.268501
1,0,"[0,0,0,0]",1,"[0,0,0,1]",state_to_state,275,0.130455
2,0,"[0,0,0,0]",2,"[0,0,1,0]",state_to_state,411,0.194972
3,0,"[0,0,0,0]",3,"[0,0,1,1]",state_to_state,11,0.005218
4,0,"[0,0,0,0]",4,"[0,1,0,0]",state_to_state,321,0.152277
...,...,...,...,...,...,...,...
116,13,"[1,1,0,1]",12,"[1,1,0,0]",state_to_state,4,0.363636
117,14,"[1,1,1,0]",END,END,state_to_END,3,0.600000
118,14,"[1,1,1,0]",6,"[0,1,1,0]",state_to_state,1,0.200000
119,14,"[1,1,1,0]",10,"[1,0,1,0]",state_to_state,1,0.200000


## 3. Changes in control ensembles and DDM parameters

This section reconstructs the 18-dimensional average CBGT activity vector for every observed network/state from the uploaded network folders, then projects each state-to-state transition through the original CCA loadings.

Required external matrices from https://doi.org/10.1371/journal.pcbi.1012966:

```text
data/source/cbgt/control/F_matrix.npy
data/source/cbgt/control/D_matrix.npy
```

In [6]:
def find_control_file(filename: str) -> Path:
    candidates = [
        SOURCE / "cbgt" / "control" / filename,
        SOURCE / "cbgt" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Expected one of:" + chr(10)
        + chr(10).join(str(path) for path in candidates)
    )

f_path = find_control_file("F_matrix.npy")
d_path = find_control_file("D_matrix.npy")

f_matrix = np.load(f_path)
d_matrix = np.load(d_path)

print("Using:")
print("F matrix:", f_path, f_matrix.shape)
print("D matrix:", d_path, d_matrix.shape)

Using:
F matrix: /Users/zhuojunyu/Desktop/CBGTPy_control/All_GitHub_Code_NEW/data/source/cbgt/F_matrix.npy (300, 18)
D matrix: /Users/zhuojunyu/Desktop/CBGTPy_control/All_GitHub_Code_NEW/data/source/cbgt/D_matrix.npy (300, 4)


In [7]:
state_activity = build_cbgt_state_activity(
    SOURCE / "cbgt" / "networks",
    n_networks=300,
)
state_activity.to_csv(
    DERIVED / "cbgt" / "state_activity_18d.csv.gz",
    index=False,
)

print("State-activity rows:", len(state_activity))

State-activity rows: 3307


In [8]:
neural_loadings, ddm_loadings = fit_control_ensembles(f_matrix, d_matrix)
control_scores = control_transition_scores(
    state_activity,
    cbgt_edges,
    neural_loadings,
    ddm_loadings,
)

# Add bit labels for readability.
control_scores.insert(
    1,
    "source_bits",
    control_scores["source"].map(lambda state: f"[{int(state):04b}]"),
)
control_scores.insert(
    3,
    "target_bits",
    control_scores["target"].map(lambda state: f"[{int(state):04b}]"),
)

# Save one combined table and two narrower tables.
write_csv(control_scores, DERIVED / "cbgt" / "control_and_ddm_transition_changes.csv")

control_columns = [
    "source",
    "source_bits",
    "target",
    "target_bits",
    "probability",
    "n_common_networks",
    "choice",
    "responsiveness",
    "pliancy",
]
ddm_columns = [
    "source",
    "source_bits",
    "target",
    "target_bits",
    "probability",
    "n_common_networks",
    "a",
    "v",
    "t",
    "z",
]

control_values = control_scores[control_columns].copy()
ddm_values = control_scores[ddm_columns].copy()

write_csv(control_values, DERIVED / "cbgt" / "control_ensemble_transition_changes.csv")
write_csv(ddm_values, DERIVED / "cbgt" / "ddm_parameter_transition_changes.csv")

print("Control ensemble changes along observed CBGT state transitions")
display(control_values.sort_values(["source", "target"]))

print("DDM parameter changes along observed CBGT state transitions")
display(ddm_values.sort_values(["source", "target"]))

Control ensemble changes along observed CBGT state transitions


,source,source_bits,target,target_bits,probability,n_common_networks,choice,responsiveness,pliancy
5,0,[0000],1,[0001],0.068580,176,-19.431045,29.042237,44.288665
6,0,[0000],2,[0010],0.071424,177,8.188142,36.013898,44.008519
14,0,[0000],3,[0011],0.023722,109,-5.217845,41.688376,52.373002
1,0,[0000],4,[0100],0.273479,292,-24.339603,31.412218,40.807639
2,0,[0000],5,[0101],0.036003,266,-34.390579,43.082977,51.046748
...,...,...,...,...,...,...,...,...,...
186,15,[1111],10,[1010],0.003228,193,29.149798,-15.770779,-11.615022
180,15,[1111],11,[1011],0.077469,127,14.233804,-10.045788,-4.128788
185,15,[1111],12,[1100],0.001291,160,0.419351,-15.593644,-11.467699
187,15,[1111],13,[1101],0.010329,152,-11.936469,-9.822391,-5.165102


DDM parameter changes along observed CBGT state transitions


,source,source_bits,target,target_bits,probability,n_common_networks,a,v,t,z
5,0,[0000],1,[0001],0.068580,176,-2.975214,-24.411418,-60.946812,10.480973
6,0,[0000],2,[0010],0.071424,177,-4.813066,6.996765,-59.626370,12.232368
14,0,[0000],3,[0011],0.023722,109,-6.637692,-8.501950,-72.584981,13.569709
1,0,[0000],4,[0100],0.273479,292,-6.521266,-28.981275,-60.563730,9.540649
2,0,[0000],5,[0101],0.036003,266,-11.848643,-40.695568,-79.206901,11.801734
...,...,...,...,...,...,...,...,...,...,...
186,15,[1111],10,[1010],0.003228,193,9.396956,32.290721,26.738892,-1.797145
180,15,[1111],11,[1011],0.077469,127,7.055224,15.267907,12.467538,-0.714172
185,15,[1111],12,[1100],0.001291,160,6.252638,0.328175,20.610314,-3.223974
187,15,[1111],13,[1101],0.010329,152,3.515959,-13.677430,8.888049,-2.202863
